# DX 704 Week 2 Project

This week's project will analyze fresh strawberry price data for a hypothetical "buy low, freeze, and sell high" business.
Strawberries show strong seasonality in their prices compared to other fruits.

![](https://ers.usda.gov/sites/default/files/_laserfiche/Charts/61401/oct14_finding_plattner_fig01.png)

Image source: https://www.ers.usda.gov/amber-waves/2014/october/seasonal-fresh-fruit-price-patterns-differ-across-commodities-the-case-of-strawberries-and-apples

You are considering a business where you buy strawberries when the prices are very low, carefully freeze them, even more carefully defrost them, and then sell them when the prices are high.
You will forecast strawberry price time series and then use them to tactically pick times to buy, freeze, and sell the strawberries.

The full project description, a template notebook, and raw data are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-02


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [16]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np

raw = pd.read_csv("strawberry-prices.tsv", sep="\t", parse_dates=["month"]).sort_values("month")

def fill_months(df, start, end):
    s = df.set_index("month").price
    return s.reindex(pd.date_range(start, end, freq="MS")).interpolate("time")

def fit_forecast(series, target_year):
    d = series.rename("price").reset_index().rename(columns={"index": "month"})
    d["year"] = d.month.dt.year
    d["m"] = d.month.dt.month
    d["dev"] = d.price - d.groupby("year").price.transform("mean")
    seasonal = d.groupby("m").dev.mean()
    seasonal = seasonal - seasonal.mean()
    level = series.iloc[-12:].mean()
    idx = pd.date_range(f"{target_year}-01-01", f"{target_year}-12-01", freq="MS")
    return pd.Series(level + seasonal.reindex(idx.month).values, index=idx)

## Part 1: Backtest Strawberry Prices

Read the provided "strawberry-prices.tsv" with data from 2020 through 2025.
This data is based on data from the U.S. Bureau of Statistics, but transformed so the ground truth is not online.
https://fred.stlouisfed.org/series/APU0000711415

Use the data for 2020 through 2024 to predict monthly prices in 2025.
Spend some time to make sure you are happy with your methodology and prediction accuracy, since you will reuse the methodology to forecast 2026 next.
Save the 2025 backtest predictions as "strawberry-backtest.tsv" with columns month and price.

In [18]:
# YOUR CHANGES HERE

train = fill_months(raw[raw.month.dt.year <= 2024], "2020-01-01", "2024-12-01")
backtest = fit_forecast(train, 2025)

pd.DataFrame({
    "month": backtest.index.strftime("%Y-%m-01"),
    "price": backtest.values.round(4),
}).to_csv("strawberry-backtest.tsv", sep="\t", index=False)

backtest

2025-01-01    4.496989
2025-02-01    4.121389
2025-03-01    3.695189
2025-04-01    3.744078
2025-05-01    3.464189
2025-06-01    3.211189
2025-07-01    3.174189
2025-08-01    3.456989
2025-09-01    3.609589
2025-10-01    3.879989
2025-11-01    4.397989
2025-12-01    4.798232
Freq: MS, dtype: float64

Please use the same format for the month column as in the training data, i.e. YYYY-MM-01.
The autograder may not be able to parse other formats.

Submit "strawberry-backtest.tsv" in Gradescope.

## Part 2: Backtest Errors

What are the mean and standard deviation of the residuals between your backtest predictions and the ground truth?

Write the mean and standard deviation to a file "backtest-accuracy.tsv" with two columns, mean and std.

In [19]:
# YOUR CHANGES HERE

actual = raw[raw.month.dt.year == 2025].set_index("month").price
residuals = actual - backtest.reindex(actual.index)
mean_res = residuals.mean()
std_res = residuals.std(ddof=1)

pd.DataFrame({"mean": [mean_res], "std": [std_res]}).to_csv("backtest-accuracy.tsv", sep="\t", index=False)

mean_res, std_res

(np.float64(-0.03680218579235008), np.float64(0.1469834869474036))

Hint: If the mean residual in your backtest is not close to zero, then your model is likely missing a systematic change and you should go back to improve it.

Submit "backtest-accuracy.tsv" in Gradescope.

## Part 3: Forecast Strawberry Prices

Use all the data from 2020 through 2025 to predict monthly prices in 2026 using the same methodology from part 1.
Make a monthly forecast for each month of 2026 and save it as "strawberry-forecast.tsv" with columns for month and price.


In [20]:
# YOUR CHANGES HERE
full = fill_months(raw, "2020-01-01", "2025-12-01")
forecast = fit_forecast(full, 2026)

pd.DataFrame({
    "month": forecast.index.strftime("%Y-%m-01"),
    "price": forecast.values.round(4),
}).to_csv("strawberry-forecast.tsv", sep="\t", index=False)

forecast

2026-01-01    4.510588
2026-02-01    4.113088
2026-03-01    3.639921
2026-04-01    3.705495
2026-05-01    3.455255
2026-06-01    3.206755
2026-07-01    3.175421
2026-08-01    3.451588
2026-09-01    3.614588
2026-10-01    3.913713
2026-11-01    4.421630
2026-12-01    4.828957
Freq: MS, dtype: float64

Submit "strawberry-forecast.tsv" in Gradescope.

## Part 4: Buy Low, Freeze and Sell High

Using your 2026 forecast, analyze the profit picking different pairs of months to buy and sell strawberries.
Maximize your profit assuming that it costs &dollar;0.20 per pint to freeze the strawberries, &dollar;0.10 per pint per month to store the frozen strawberries and there is a 10% price discount from selling previously frozen strawberries.
So, if you buy a pint of strawberies for &dollar;1, freeze them, and sell them for &dollar;2 three months after buying them, then the profit is &dollar;2 * 0.9 - &dollar;1 - &dollar;0.20 - &dollar;0.10 * 3 = &dollar;0.30 per pint.
To evaluate a given pair of months, assume that you can invest &dollar;1,000,000 to cover all costs, and that you buy as many pints of strawberries as possible.

Write the results of your analysis to a file "timings.tsv" with columns for the buy_month, sell_month, pints_purchased, and expected_profit.

In [21]:
# YOUR CHANGES HERE
CAPITAL = 1_000_000.0
FREEZE = 0.20
STORE = 0.10
DISCOUNT = 0.10

rows = []
for i, buy_month in enumerate(forecast.index):
    for j, sell_month in enumerate(forecast.index):
        if j <= i:
            continue
        months = j - i
        cost = forecast.iloc[i] + FREEZE + STORE * months
        pints = int(CAPITAL // cost)
        unit_profit = forecast.iloc[j] * (1 - DISCOUNT) - cost
        rows.append((
            buy_month.strftime("%Y-%m-01"),
            sell_month.strftime("%Y-%m-01"),
            pints,
            round(unit_profit * pints, 2),
        ))

timings = pd.DataFrame(rows, columns=["buy_month", "sell_month", "pints_purchased", "expected_profit"])
timings.to_csv("timings.tsv", sep="\t", index=False)

timings.sort_values("expected_profit", ascending=False).head()

,buy_month,sell_month,pints_purchased,expected_profit
55,2026-07-01,2026-12-01,258036,121442.02
50,2026-06-01,2026-12-01,249578,84683.43
59,2026-08-01,2026-12-01,246816,72680.69
62,2026-09-01,2026-12-01,243037,56256.54
54,2026-07-01,2026-11-01,264871,54045.81


Submit "timings.tsv" in Gradescope.

## Part 5: Strategy Check

What is the best profit scenario according to your previous timing analysis?
How much does that profit change if the sell price is off by one standard deviation from your backtest analysis?
(Variation in the sell price is more dangerous because you can see the buy price before fully committing.)

Write the results to a file "check.tsv" with columns `best_profit` and `one_std_profit`.
To be clear, `one_std_profit` should be the number of pints bought in your best profit scenario times your backtested standard deviation of the residual.
This represents the standard deviation in revenue when selling if you explicitly assume that you buy according to the best profit scenario and your backtest standard deviation is representative of the future prices.

In [22]:
# YOUR CHANGES HERE
best = timings.loc[timings.expected_profit.idxmax()]
best_profit = float(best.expected_profit)
one_std_profit = float(best.pints_purchased) * std_res

pd.DataFrame({
    "best_profit": [round(best_profit, 2)],
    "one_std_profit": [round(one_std_profit, 2)],
}).to_csv("check.tsv", sep="\t", index=False)

best

buy_month          2026-07-01
sell_month         2026-12-01
pints_purchased        258036
expected_profit     121442.02
Name: 55, dtype: object

Submit "check.tsv" in Gradescope.

## Part 6: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.